# Plajah Motorsport: Apex — Ironport Buildings Photoreal 2K PBR + Plajah Native 3D Modeler (Colab Pro+ GPU)

**100% Plajah-Native Pipeline (Zero Blender!)** designed for **Colab Pro+ A100 / L4 GPU**:

### Stage 1 — Neural 2K Photoreal PBR Engine (All 60 Material Sets: 24 Shared + 36 Hero Facades)
- **Real-ESRGAN x4plus (`spandrel`)**: Crisp 2048×2048 super-resolution preserving exact hero facade pixel layouts
- **Marigold IID Appearance (`prs-eth/marigold-iid-appearance-v1-1`)**: Frequency-separated neural delighting (removes harsh baked shadows while preserving razor-sharp 2K brick/stone grain) + real per-pixel physical roughness & metalness
- **Marigold Normals (`prs-eth/marigold-normals-v1-1`)**: True 3D architectural surface normals (auto-verified OpenGL +Y-up!) RNM-blended with 2K tactile micro-normals + **Toksvig variance-to-roughness anti-sparkle filtering**
- **Depth Anything V2 Large (`depth-anything/Depth-Anything-V2-Large-hf`)**: Metric architectural relief height maps (`_hgt`) for real-time Parallax Occlusion Mapping (POM) + Stage 2 3D Hero Relief Modeling
- **Calibrated ORM Packing (`_orm`)**: `R = Cavity AO`, `G = Calibrated Roughness` (surgical window-glass separation on hero facades), `B = Physical Metalness`

### Stage 2 — Plajah Native Architectural 3D Modeler + PyTorch CUDA Raytraced AO Baker (All 186 Buildings)
- Runs Plajah's own **`plajah_arch_modeler.py`** (bundled inside `ironport_v4_inputs.zip`, zero Blender!)
- Replaces flat 2-triangle `*_front` hero facade quads with **Depth-Guided 3D Architectural Relief Meshes** (recessed windows, extruded stone piers, carved arches, UV-locked)
- Upgrades cylindrical skyscrapers (`ironport-towers`, 572 tris -> ~12k tris) with real 3D vertical fins, floor spandrels, and crown collars
- Adds archetype-aware 3D architectural geometry across all **36 Heroes + 150 Generics** (beveled corner pilasters, multi-step molded cornices, parapets, belt courses, overhanging house eaves/porches/chimneys, mechanical penthouses)
- **PyTorch CUDA 64-Ray Hemisphere Vertex AO Baker**: bakes crisp photoreal contact shadows directly into glTF `COLOR_0` at **zero runtime FPS cost**

**Before running:** Set **Runtime → Change runtime type → `A100 GPU` (recommended) or `L4 GPU`**, and upload `ironport_v4_inputs.zip` to `/content/` (or Google Drive).

In [ ]:
# Cell 1: Verify A100/L4 GPU and install pure wheel dependencies (Zero Blender, zero CUDA compilation!)
import os, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU attached! Go to Runtime -> Change runtime type -> A100 GPU or L4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM), PyTorch {torch.__version__}')

pkgs = [
    'diffusers>=0.32.0', 'transformers>=4.45.0', 'accelerate', 'safetensors',
    'timm', 'spandrel', 'opencv-python-headless', 'scipy', 'Pillow',
    'fast-simplification', 'matplotlib', 'tqdm'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

import diffusers, transformers, spandrel, cv2, fast_simplification
print('diffusers:', diffusers.__version__, '| transformers:', transformers.__version__, '| fast_simplification OK')

In [ ]:
# Cell 2: Locate & unpack ironport_v4_inputs.zip (supports /content/ upload OR Google Drive)
import glob, json, os, shutil, sys, zipfile

USE_GOOGLE_DRIVE = False  # Set True to read/write zips directly on Google Drive (/content/drive/MyDrive/plajah_apex_v4/)
INPUT_ZIP = None          # Leave None to auto-detect /content/ironport_v4*_inputs.zip

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/plajah_apex_v4'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    if INPUT_ZIP is None:
        cands = sorted(glob.glob(os.path.join(DRIVE_DIR, '*ironport*inputs*.zip')), key=os.path.getmtime, reverse=True)
        if cands:
            INPUT_ZIP = cands[0]

if INPUT_ZIP is None:
    cands = sorted(glob.glob('/content/*ironport*inputs*.zip'), key=os.path.getmtime, reverse=True)
    if cands:
        INPUT_ZIP = cands[0]
    else:
        from google.colab import files
        print('Upload ironport_v4_inputs.zip via the Files sidebar or prompt...')
        uploaded = files.upload()
        INPUT_ZIP = os.path.join('/content', next(iter(uploaded.keys())))

assert os.path.isfile(INPUT_ZIP), f'Missing input zip: {INPUT_ZIP}'
print(f'Input archive: {INPUT_ZIP} ({os.path.getsize(INPUT_ZIP)/1e6:.2f} MB)')

WORK_IN = '/content/ironport_v4_in'
WORK_OUT = '/content/ironport_v4_out'
if os.path.isdir(WORK_IN):
    shutil.rmtree(WORK_IN)
os.makedirs(WORK_IN, exist_ok=True)
os.makedirs(WORK_OUT, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP, 'r') as zf:
    zf.extractall(WORK_IN)

with open(os.path.join(WORK_IN, 'manifest.json'), 'r', encoding='utf-8') as f:
    MANIFEST = json.load(f)

if WORK_IN not in sys.path:
    sys.path.insert(0, WORK_IN)

import plajah_arch_modeler
print('Loaded Plajah Native Architectural Modeler:', plajah_arch_modeler.__file__)
print('Batch counts:', json.dumps(MANIFEST['counts'], indent=2))

In [ ]:
# Cell 3: Stage 1 — Colab GPU 2K Neural PBR Engine (Real-ESRGAN + Marigold IID + Marigold Normals + Depth Anything V2)
import gc, json, math, os, time, urllib.request
import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import diffusers
from transformers import pipeline as hf_pipeline
import spandrel

# Config knobs
ENABLE_ESRGAN_2K = True       # Real-ESRGAN x4plus tiled 2K super-resolution
MARIGOLD_STEPS = 4            # 4 denoising steps (optimal quality/speed for Marigold v1-1 LCM)
MARIGOLD_ENSEMBLE = 2         # 2-pass ensemble for clean, low-noise normals & intrinsics
DELIGHT_STRENGTH = 0.62       # Preserves crisp 2K stone/brick texture while lifting harsh baked shadows
SEAMLESS_BORDER_PX = 96       # Cross-fade seam band on 2048px shared tileable textures
FORCE_REBUILD_TEX = False     # Set True to overwrite cached outputs

TEX_OUT = os.path.join(WORK_OUT, 'textures-v4')
os.makedirs(os.path.join(TEX_OUT, 'shared'), exist_ok=True)
os.makedirs(os.path.join(TEX_OUT, 'hero-facades'), exist_ok=True)

# --- 3A. Download & cache Real-ESRGAN x4plus weights ---
ESRGAN_PATH = '/content/RealESRGAN_x4plus.pth'
if ENABLE_ESRGAN_2K and not os.path.isfile(ESRGAN_PATH):
    print('Downloading RealESRGAN_x4plus.pth (~67 MB)...')
    try:
        urllib.request.urlretrieve(
            'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
            ESRGAN_PATH,
        )
    except Exception as e:
        print('Real-ESRGAN download skipped (using Lanczos+micro-sharpen fallback):', e)
        ENABLE_ESRGAN_2K = False

def upscale_2k_tiled(rgb_u8: np.ndarray, esr_model=None, target_size: int = 2048) -> np.ndarray:
    h, w = rgb_u8.shape[:2]
    if esr_model is None or w >= target_size:
        out = cv2.resize(rgb_u8, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
        blur = cv2.GaussianBlur(out, (0, 0), 1.0)
        return np.clip(cv2.addWeighted(out, 1.18, blur, -0.18, 0), 0, 255).astype(np.uint8)
    # Tiled 512px inference with 32px overlap -> 4x -> crisp Lanczos downsample to 2048px
    tile, ov, scale = 512, 32, 4
    out_h, out_w = h * scale, w * scale
    acc = np.zeros((out_h, out_w, 3), dtype=np.float32)
    wt = np.zeros((out_h, out_w, 1), dtype=np.float32)
    with torch.no_grad():
        for y0 in range(0, h, tile - ov):
            y1 = min(h, y0 + tile)
            y0 = max(0, y1 - tile)
            for x0 in range(0, w, tile - ov):
                x1 = min(w, x0 + tile)
                x0 = max(0, x1 - tile)
                patch = rgb_u8[y0:y1, x0:x1]
                t_in = torch.from_numpy(patch).permute(2, 0, 1)[None].float().div(255.0).cuda().half()
                t_out = esr_model(t_in)[0].float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255.0
                ph, pw = t_out.shape[:2]
                wy = np.hanning(ph + 2)[1:-1, None]
                wx = np.hanning(pw + 2)[None, 1:-1]
                w2 = (np.maximum(0.05, wy * wx))[:, :, None].astype(np.float32)
                acc[y0*scale:y1*scale, x0*scale:x1*scale] += t_out * w2
                wt[y0*scale:y1*scale, x0*scale:x1*scale] += w2
    up4 = np.clip(acc / np.maximum(wt, 1e-5), 0, 255).astype(np.uint8)
    return cv2.resize(up4, (target_size, target_size), interpolation=cv2.INTER_AREA)

def rnm_blend(n_base: np.ndarray, n_detail: np.ndarray) -> np.ndarray:
    """Reoriented Normal Mapping blend of two unit normal maps in [-1, 1] (OpenGL +Y up, +Z out)."""
    t = n_base.copy()
    t[:, :, 2] += 1.0
    u = n_detail.copy()
    u[:, :, 0] *= -1.0
    u[:, :, 1] *= -1.0
    dot = np.sum(t * u, axis=-1, keepdims=True)
    r = t * dot / np.maximum(t[:, :, 2:3], 1e-4) - u
    l = np.linalg.norm(r, axis=-1, keepdims=True)
    return r / np.maximum(l, 1e-6)

def toksvig_roughness(rgh01: np.ndarray, nrm_unit: np.ndarray, strength: float = 0.45) -> np.ndarray:
    """Fold unresolved sub-pixel normal variance into physical roughness (anti-sparkle mip filtering)."""
    n_avg = cv2.GaussianBlur(nrm_unit.astype(np.float32), (0, 0), 1.35)
    n_len = np.clip(np.linalg.norm(n_avg, axis=-1), 0.25, 1.0)
    variance = np.clip((1.0 - n_len) / np.maximum(n_len, 0.25), 0.0, 0.65)
    return np.clip(np.sqrt(rgh01 * rgh01 + variance * strength), 0.03, 0.98)

def make_seamless_maps(maps: dict, border_px: int = 96) -> dict:
    """Apply identical periodic cross-fade across all maps of a shared tileable PBR set."""
    h, w = next(iter(maps.values())).shape[:2]
    b = min(border_px, h // 8, w // 8)
    wt = np.arange(b, dtype=np.float32) / float(b)
    wt = wt * wt * (3.0 - 2.0 * wt)
    out = {}
    for k, arr in maps.items():
        a = arr.astype(np.float32)
        is_2d = (a.ndim == 2)
        if is_2d:
            a = a[:, :, None]
        # Horizontal wrap
        wx = wt[None, :, None]
        ax = a[:, :w - b, :].copy()
        ax[:, :b, :] = a[:, :b, :] * wx + a[:, w - b:, :] * (1.0 - wx)
        # Vertical wrap
        wy = wt[:, None, None]
        ay = ax[:h - b, :, :].copy()
        ay[:b, :, :] = ax[:b, :, :] * wy + ax[h - b:, :, :] * (1.0 - wy)
        res = cv2.resize(ay, (w, h), interpolation=cv2.INTER_CUBIC)
        if res.ndim == 2 and not is_2d:
            res = res[:, :, None]
        if is_2d and res.ndim == 3:
            res = res[:, :, 0]
        out[k] = res
    return out

# --- 3B. Pass 1: Super-resolve all 60 textures to 2048x2048 + Depth Anything V2 Large ---
sr_cache = {}
depth_cache = {}

esr_model = None
if ENABLE_ESRGAN_2K:
    print('Loading Real-ESRGAN x4plus on CUDA...')
    esr_model = spandrel.ModelLoader().load_from_file(ESRGAN_PATH).cuda().eval().half()

print('Loading Depth Anything V2 Large on CUDA...')
depth_pipe = hf_pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Large-hf', device=0, torch_dtype=torch.float16)

t0 = time.time()
tex_items = MANIFEST['textures']
for idx, item in enumerate(tex_items):
    stem = item['stem']
    grp = item['group']
    out_dir = os.path.join(TEX_OUT, grp)
    done_flag = os.path.join(out_dir, f'{stem}_orm.jpg')
    if os.path.isfile(done_flag) and not FORCE_REBUILD_TEX:
        continue
    src_path = os.path.join(WORK_IN, item['src'])
    rgb = np.array(Image.open(src_path).convert('RGB'))
    rgb2k = upscale_2k_tiled(rgb, esr_model, target_size=2048)
    sr_cache[stem] = rgb2k
    # Depth Anything V2 Large (runs on 1024px for sharp architectural depth)
    im_d = Image.fromarray(cv2.resize(rgb2k, (1024, 1024), interpolation=cv2.INTER_AREA))
    with torch.no_grad():
        d_res = depth_pipe(im_d)
    d_t = d_res['predicted_depth']
    d_raw = d_t.detach().float().cpu().numpy().astype(np.float32) if hasattr(d_t, 'detach') else np.asarray(d_t, dtype=np.float32)
    d_raw = cv2.resize(d_raw, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    p2, p98 = float(np.percentile(d_raw, 2)), float(np.percentile(d_raw, 98))
    depth_cache[stem] = np.clip((d_raw - p2) / max(1e-5, p98 - p2), 0.0, 1.0)
    if (idx + 1) % 10 == 0 or idx + 1 == len(tex_items):
        print(f'  [SR + Depth] {idx + 1}/{len(tex_items)} ({time.time() - t0:.1f}s)', flush=True)

del esr_model, depth_pipe
gc.collect(); torch.cuda.empty_cache()

# --- 3C. Pass 2: Marigold IID Appearance (Neural Delighting + Roughness + Metalness) ---
iid_cache = {}
if sr_cache:
    print('Loading prs-eth/marigold-iid-appearance-v1-1 (fp16) on CUDA...')
    iid_pipe = diffusers.MarigoldIntrinsicsPipeline.from_pretrained(
        'prs-eth/marigold-iid-appearance-v1-1', variant='fp16', torch_dtype=torch.float16
    ).to('cuda')
    iid_pipe.set_progress_bar_config(disable=True)
    t1 = time.time()
    for idx, (stem, rgb2k) in enumerate(sr_cache.items()):
        im_in = Image.fromarray(cv2.resize(rgb2k, (768, 768), interpolation=cv2.INTER_AREA))
        with torch.no_grad():
            out = iid_pipe(im_in, num_inference_steps=MARIGOLD_STEPS, ensemble_size=MARIGOLD_ENSEMBLE)
        vis = iid_pipe.image_processor.visualize_intrinsics(out.prediction, iid_pipe.target_properties)[0]
        alb = np.asarray(vis['albedo'].convert('RGB'), dtype=np.float32) / 255.0
        rgh = np.asarray(vis['roughness'].convert('L'), dtype=np.float32) / 255.0
        mtl = np.asarray(vis['metallicity'].convert('L'), dtype=np.float32) / 255.0
        iid_cache[stem] = (alb, rgh, mtl)
        if (idx + 1) % 10 == 0 or idx + 1 == len(sr_cache):
            print(f'  [Marigold IID] {idx + 1}/{len(sr_cache)} ({time.time() - t1:.1f}s)', flush=True)
    del iid_pipe
    gc.collect(); torch.cuda.empty_cache()

# --- 3D. Pass 3: Marigold Surface Normals + Fused Physical PBR Calibration & Export ---
qa_samples = []
if sr_cache:
    print('Loading prs-eth/marigold-normals-v1-1 (fp16) on CUDA...')
    nrm_pipe = diffusers.MarigoldNormalsPipeline.from_pretrained(
        'prs-eth/marigold-normals-v1-1', variant='fp16', torch_dtype=torch.float16
    ).to('cuda')
    nrm_pipe.set_progress_bar_config(disable=True)
    t2 = time.time()
    for idx, item in enumerate(tex_items):
        stem = item['stem']
        if stem not in sr_cache:
            continue
        grp = item['group']
        is_hero = (grp == 'hero-facades')
        rgb2k = sr_cache[stem]
        rgb_f = rgb2k.astype(np.float32) / 255.0
        depth1k = depth_cache[stem]
        iid_alb, iid_rgh, iid_mtl = iid_cache[stem]

        # Predict neural surface normals at 768px
        im_in = Image.fromarray(cv2.resize(rgb2k, (768, 768), interpolation=cv2.INTER_AREA))
        with torch.no_grad():
            n_out = nrm_pipe(im_in, num_inference_steps=MARIGOLD_STEPS, ensemble_size=MARIGOLD_ENSEMBLE)
        n_raw = n_out.prediction
        if hasattr(n_raw, 'detach'):
            n_raw = n_raw.detach().float().cpu().numpy()
        n_pred = np.squeeze(np.asarray(n_raw, dtype=np.float32))
        if n_pred.ndim == 3 and n_pred.shape[0] == 3:
            n_pred = np.transpose(n_pred, (1, 2, 0))
        n_pred = cv2.resize(n_pred.astype(np.float32), (1024, 1024), interpolation=cv2.INTER_CUBIC)
        # Auto-verify OpenGL (+X right, +Y up, +Z out) against Depth Anything V2 gradients!
        gx_d = cv2.Sobel(depth1k, cv2.CV_32F, 1, 0, ksize=3)
        gy_d = cv2.Sobel(depth1k, cv2.CV_32F, 0, 1, ksize=3)
        if float(np.mean(n_pred[:, :, 0] * (-gx_d))) < 0.0:
            n_pred[:, :, 0] *= -1.0
        if float(np.mean(n_pred[:, :, 1] * (+gy_d))) < 0.0:
            n_pred[:, :, 1] *= -1.0
        if float(np.mean(n_pred[:, :, 2])) < 0.0:
            n_pred[:, :, 2] *= -1.0
        n_pred[:, :, 2] = np.maximum(n_pred[:, :, 2], 0.15)
        n_pred /= np.maximum(np.linalg.norm(n_pred, axis=-1, keepdims=True), 1e-6)

        # 1) Frequency-separated 2K neural delighting (preserves sharp 2K brick/stone grain!)
        alb_2k = cv2.resize(iid_alb, (2048, 2048), interpolation=cv2.INTER_CUBIC)
        orig_low = cv2.GaussianBlur(rgb_f, (0, 0), 22.0)
        alb_low = cv2.GaussianBlur(alb_2k, (0, 0), 22.0)
        ratio = np.clip((alb_low + 0.04) / (orig_low + 0.04), 0.68, 1.42)
        delit_2k = np.clip(rgb_f * (1.0 - DELIGHT_STRENGTH + DELIGHT_STRENGTH * ratio), 0.0, 1.0)

        # 2) High-frequency 2K tactile micro-normal + cavity from delit luminance (mortar, stone pores, bolts)
        lum2k = cv2.cvtColor((delit_2k * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        hf = lum2k - cv2.GaussianBlur(lum2k, (0, 0), 9.0)
        hf = cv2.GaussianBlur(hf, (0, 0), 0.85)
        hf_norm = hf / (float(hf.std()) + 1e-5)
        relief_k = float(item.get('relief', 1.0))
        mgx = cv2.Sobel(hf_norm, cv2.CV_32F, 1, 0, ksize=3) / 8.0
        mgy = cv2.Sobel(hf_norm, cv2.CV_32F, 0, 1, ksize=3) / 8.0
        # OpenGL +Y up micro-normal: nx = -mgx, ny = +mgy, nz = 1
        micro_n = np.stack([-mgx * 0.42 * relief_k, +mgy * 0.42 * relief_k, np.ones_like(mgx)], axis=-1)
        micro_n /= np.maximum(np.linalg.norm(micro_n, axis=-1, keepdims=True), 1e-6)

        # RNM blend neural 3D macro normals (2K) + 2K micro-normals
        n_macro_2k = cv2.resize(n_pred, (2048, 2048), interpolation=cv2.INTER_CUBIC)
        n_macro_2k /= np.maximum(np.linalg.norm(n_macro_2k, axis=-1, keepdims=True), 1e-6)
        # Soften neural normal extremes slightly so architecture stays crisp, not ballooned
        n_macro_2k[:, :, :2] *= (0.78 * relief_k)
        n_macro_2k[:, :, 2] = np.maximum(n_macro_2k[:, :, 2], 0.25)
        n_macro_2k /= np.maximum(np.linalg.norm(n_macro_2k, axis=-1, keepdims=True), 1e-6)
        n_final_2k = rnm_blend(n_macro_2k, micro_n)

        # 3) Fused architectural Height map (Depth Anything V2 macro + mortar/recess micro)
        hf_1k = cv2.resize(np.clip(0.5 + hf_norm * 0.16, 0.0, 1.0), (1024, 1024), interpolation=cv2.INTER_AREA)
        # Remove slow perspective gradient tilt from depth map (stronger periodic flattening for tileable shared walls)
        if item.get('seamless', False):
            _t3 = np.tile(depth1k, (3, 3))
            _h, _w = depth1k.shape
            d_flat = depth1k - 0.90 * cv2.GaussianBlur(_t3, (0, 0), 42.0)[_h:2*_h, _w:2*_w]
        else:
            d_flat = depth1k - 0.35 * cv2.GaussianBlur(depth1k, (0, 0), 64.0)
        dp2, dp98 = float(np.percentile(d_flat, 2)), float(np.percentile(d_flat, 98))
        d_norm = np.clip((d_flat - dp2) / max(1e-5, dp98 - dp2), 0.0, 1.0)
        hgt_1k = np.clip(0.74 * d_norm + 0.26 * hf_1k, 0.0, 1.0)

        # 4) Physically calibrated ORM (R = Cavity AO, G = Toksvig Roughness, B = Metalness)
        ao_macro = np.clip(0.84 + (d_norm - cv2.GaussianBlur(d_norm, (0, 0), 18.0)) * 0.85, 0.42, 1.0)
        ao_micro = np.clip(0.88 + cv2.resize(hf_norm, (1024, 1024), interpolation=cv2.INTER_AREA) * 0.08, 0.55, 1.0)
        ao_1k = np.clip(ao_macro * ao_micro, 0.36, 1.0)

        rgh_nn = cv2.resize(iid_rgh, (1024, 1024), interpolation=cv2.INTER_CUBIC)
        mtl_nn = cv2.resize(iid_mtl, (1024, 1024), interpolation=cv2.INTER_CUBIC)
        r_lo, r_hi = item.get('rgh', [0.05, 0.92])
        m_lo, m_hi = item.get('mtl', [0.0, 0.85])

        if not is_hero:
            # Anchor Marigold IID relative variation into physically valid range for this architectural material
            r_p5, r_p95 = float(np.percentile(rgh_nn, 5)), float(np.percentile(rgh_nn, 95))
            r_rel = np.clip((rgh_nn - r_p5) / max(0.08, r_p95 - r_p5), 0.0, 1.0)
            rgh_1k = r_lo + r_rel * (r_hi - r_lo)
            # Crevices & mortar are slightly rougher
            rgh_1k = np.clip(rgh_1k + (1.0 - ao_1k) * 0.12, r_lo, r_hi)
            m_p5, m_p95 = float(np.percentile(mtl_nn, 5)), float(np.percentile(mtl_nn, 95))
            m_rel = np.clip((mtl_nn - m_p5) / max(0.08, m_p95 - m_p5), 0.0, 1.0)
            mtl_1k = m_lo + m_rel * (m_hi - m_lo)
            # Curtain-wall glass panes: smooth panes, metallic mullions
            if item.get('class') == 'glass':
                rgb1k = cv2.resize(rgb_f, (1024, 1024), interpolation=cv2.INTER_AREA)
                is_mull = (hgt_1k > 0.62) | (np.max(rgb1k, axis=-1) - np.min(rgb1k, axis=-1) < 0.06) & (np.mean(rgb1k, axis=-1) > 0.55)
                rgh_1k = np.where(is_mull, np.clip(rgh_1k, 0.22, 0.38), 0.04)
                mtl_1k = np.where(is_mull, 0.82, 0.14)
        else:
            # Hero architectural facade: combine Marigold IID + Depth cavity + optical glass cue
            rgb1k = cv2.resize(rgb_f, (1024, 1024), interpolation=cv2.INTER_AREA)
            maxc = np.max(rgb1k, axis=-1)
            minc = np.min(rgb1k, axis=-1)
            sat = (maxc - minc) / np.maximum(maxc, 0.05)
            lum1 = 0.299 * rgb1k[:, :, 0] + 0.587 * rgb1k[:, :, 1] + 0.114 * rgb1k[:, :, 2]
            cool_glass = np.clip((rgb1k[:, :, 2] - rgb1k[:, :, 0] + 0.02) * 5.0, 0.0, 1.0)
            recessed = np.clip((0.55 - d_norm) * 2.5, 0.0, 1.0)
            nn_smooth = np.clip((0.52 - rgh_nn) * 2.4, 0.0, 1.0)
            # Smooth, recessed, dark-or-cool low-normal-variance panes = dielectric architectural window glass
            glass_prob = np.clip(
                (0.45 * nn_smooth + 0.30 * recessed + 0.25 * np.maximum(cool_glass, 1.0 - np.clip(lum1 * 2.2, 0, 1)))
                * (1.0 - np.clip((sat - 0.38) * 3.0, 0.0, 0.85)),
                0.0, 1.0,
            )
            glass_mask = (glass_prob > 0.56).astype(np.float32)
            glass_mask = cv2.GaussianBlur(glass_mask, (0, 0), 0.8)
            rgh_1k = np.clip(0.32 + 0.62 * rgh_nn + (1.0 - ao_1k) * 0.10, 0.24, 0.94)
            rgh_1k = rgh_1k * (1.0 - glass_mask) + 0.045 * glass_mask
            mtl_1k = np.clip(mtl_nn * 0.85, 0.0, 0.85) * (1.0 - glass_mask) + 0.12 * glass_mask
            # Flatten normal on glass panes so reflections stay optically crisp
            gm2k = cv2.resize(glass_mask, (2048, 2048), interpolation=cv2.INTER_LINEAR)[:, :, None]
            flat_n = np.zeros_like(n_final_2k); flat_n[:, :, 2] = 1.0
            n_final_2k = n_final_2k * (1.0 - 0.92 * gm2k) + flat_n * (0.92 * gm2k)
            n_final_2k /= np.maximum(np.linalg.norm(n_final_2k, axis=-1, keepdims=True), 1e-6)

        # Apply Toksvig variance-to-roughness filtering (eliminates shimmering bricks/roofs at speed!)
        n_1k = cv2.resize(n_final_2k, (1024, 1024), interpolation=cv2.INTER_AREA)
        n_1k /= np.maximum(np.linalg.norm(n_1k, axis=-1, keepdims=True), 1e-6)
        rgh_1k = np.where(rgh_1k < 0.12, rgh_1k, toksvig_roughness(rgh_1k, n_1k, strength=0.40))

        # 5) Seamless periodic edge blending for shared tileable materials (all maps together!)
        nrm_enc_2k = np.clip(n_final_2k * 0.5 + 0.5, 0.0, 1.0)
        if item.get('seamless', False):
            packed_2k = make_seamless_maps({'base': delit_2k, 'nrm': nrm_enc_2k}, border_px=SEAMLESS_BORDER_PX)
            delit_2k = np.clip(packed_2k['base'], 0.0, 1.0)
            nrm_v = packed_2k['nrm'] * 2.0 - 1.0
            nrm_v /= np.maximum(np.linalg.norm(nrm_v, axis=-1, keepdims=True), 1e-6)
            nrm_enc_2k = np.clip(nrm_v * 0.5 + 0.5, 0.0, 1.0)
            packed_1k = make_seamless_maps({'ao': ao_1k, 'rgh': rgh_1k, 'mtl': mtl_1k, 'hgt': hgt_1k}, border_px=SEAMLESS_BORDER_PX // 2)
            ao_1k = np.clip(packed_1k['ao'], 0.30, 1.0)
            rgh_1k = np.clip(packed_1k['rgh'], 0.03, 0.98)
            mtl_1k = np.clip(packed_1k['mtl'], 0.0, 1.0)
            hgt_1k = np.clip(packed_1k['hgt'], 0.0, 1.0)

        # 6) Save full v4 texture suite (2048 base + 1024/2048 nrm + 1024 rgh/orm/hgt)
        out_dir = os.path.join(TEX_OUT, grp)
        pbr_sz = int(item.get('targetPbrRes', 1024))
        nrm_save = nrm_enc_2k if pbr_sz >= 2048 else cv2.resize(nrm_enc_2k, (pbr_sz, pbr_sz), interpolation=cv2.INTER_AREA)
        orm_1k = np.stack([ao_1k, rgh_1k, mtl_1k], axis=-1)

        Image.fromarray((delit_2k * 255).astype(np.uint8)).save(os.path.join(out_dir, f'{stem}_base.jpg'), quality=92, optimize=True)
        Image.fromarray((nrm_save * 255).astype(np.uint8)).save(os.path.join(out_dir, f'{stem}_nrm.jpg'), quality=94, optimize=True)
        Image.fromarray((rgh_1k * 255).astype(np.uint8)).save(os.path.join(out_dir, f'{stem}_rgh.jpg'), quality=90, optimize=True)
        Image.fromarray((orm_1k * 255).astype(np.uint8)).save(os.path.join(out_dir, f'{stem}_orm.jpg'), quality=92, optimize=True)
        Image.fromarray((hgt_1k * 255).astype(np.uint8)).save(os.path.join(out_dir, f'{stem}_hgt.jpg'), quality=92, optimize=True)

        if stem in ('tex-shared-01-red-brick', 'tex-hero-07-the-guardian', 'tex-hero-29-old-cathedral', 'tex-hero-01-ironport-towers', 'tex-shared-04-limestone'):
            qa_samples.append((stem, rgb_f, delit_2k, nrm_enc_2k, orm_1k, hgt_1k))

        if (idx + 1) % 10 == 0 or idx + 1 == len(tex_items):
            print(f'  [Normals + Calibrated PBR] {idx + 1}/{len(tex_items)} ({time.time() - t2:.1f}s)', flush=True)

    del nrm_pipe
    sr_cache.clear(); depth_cache.clear(); iid_cache.clear()
    gc.collect(); torch.cuda.empty_cache()

# Render inline QA Contact Sheet (Original | 2K Delit Albedo | Neural Normal | Calibrated ORM | Depth Height)
if qa_samples:
    fig, axes = plt.subplots(len(qa_samples), 5, figsize=(16, 3.2 * len(qa_samples)), facecolor='#12141c')
    if len(qa_samples) == 1:
        axes = [axes]
    titles = ['Input v3', 'v4 2K Delit Albedo', 'v4 Neural+Micro Normal', 'v4 ORM (R:AO G:Rgh B:Mtl)', 'v4 Depth Height (POM)']
    for row, (stem, orig, delit, nrm, orm, hgt) in enumerate(qa_samples):
        imgs = [orig, delit, nrm, orm, hgt]
        for col, im_a in enumerate(imgs):
            ax = axes[row][col]
            ax.axis('off')
            ax.imshow(cv2.resize(im_a, (384, 384)) if im_a.ndim == 3 else cv2.resize(im_a, (384, 384)), cmap=('gray' if im_a.ndim == 2 else None))
            ax.set_title(f'{stem}\n{titles[col]}' if col == 0 else titles[col], color='white', fontsize=9)
    plt.tight_layout()
    pbr_qa = os.path.join(WORK_OUT, 'pbr_qa_preview.png')
    plt.savefig(pbr_qa, dpi=130, facecolor='#12141c')
    plt.show()
    print('Stage 1 PBR QA saved:', pbr_qa)

In [ ]:
# Cell 4: Stage 2 — Plajah Native Architectural 3D Modeler + PyTorch CUDA 64-Ray Vertex AO Baker (Zero Blender!)
import gc, importlib, json, os, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import plajah_arch_modeler
importlib.reload(plajah_arch_modeler)

AO_SAMPLES = 64          # 64 cosine-weighted hemisphere rays/vertex on A100/L4 CUDA
HERO_RELIEF_TRIS = 7500  # Adaptive target triangles per displaced 3D hero facade relief

MOD_IN = os.path.join(WORK_IN, 'models-v3')
MOD_OUT = os.path.join(WORK_OUT, 'models-v4')
TEX_V4 = os.path.join(WORK_OUT, 'textures-v4')
os.makedirs(os.path.join(MOD_OUT, 'hero'), exist_ok=True)
os.makedirs(os.path.join(MOD_OUT, 'generic'), exist_ok=True)

results = []
t0 = time.time()
models_list = MANIFEST['models']
print(f'Running Plajah Native Architectural Modeler on {len(models_list)} buildings ({AO_SAMPLES}-ray CUDA AO)...')

for i, m in enumerate(models_list):
    kind, slug = m['kind'], m['slug']
    in_gltf = os.path.join(WORK_IN, m['gltf'])
    out_gltf = os.path.join(MOD_OUT, kind, f'{slug}_lod0.gltf')
    res = plajah_arch_modeler.upgrade_building(
        in_gltf, out_gltf, kind,
        tex_v4_dir=TEX_V4,
        device='cuda',
        ao_samples=AO_SAMPLES,
        hero_target_tris=HERO_RELIEF_TRIS,
    )
    results.append(res)
    if kind == 'hero' or (i + 1) % 15 == 0 or (i + 1) == len(models_list):
        print(
            f"[{i+1:3d}/{len(models_list)}] {res['id']:38s} "
            f"{res['tris_before']:6d} -> {res['tris_after']:6d} tris "
            f"(relief+{res['hero_relief_added']}, cyl+{res['cyl_added']}, arch+{res['arch_added']}, ao={res['mean_ao']:.2f}) "
            f"[{res['seconds']:.2f}s]",
            flush=True,
        )

tot_before = sum(r['tris_before'] for r in results)
tot_after = sum(r['tris_after'] for r in results)
geo_report = {
    'gpu': torch.cuda.get_device_name(0),
    'modeler': 'Plajah Native Architectural Modeler v4 (Zero Blender)',
    'ao_samples': AO_SAMPLES,
    'count': len(results),
    'total_tris_before': tot_before,
    'total_tris_after': tot_after,
    'ratio': round(tot_after / max(1, tot_before), 2),
    'seconds_total': round(time.time() - t0, 1),
    'results': results,
}
with open(os.path.join(WORK_OUT, 'geometry_report.json'), 'w', encoding='utf-8') as f:
    json.dump(geo_report, f, indent=2)
print(f"\nSTAGE 2 COMPLETE in {geo_report['seconds_total']}s: {tot_before:,} -> {tot_after:,} tris ({geo_report['ratio']}x geometry density!)")

# Render Before (v3) vs After (v4 + CUDA AO) 3D Point-Shaded Visual QA Sheet
showcase = [
    ('hero', 'ironport-towers'),
    ('hero', 'the-guardian'),
    ('hero', 'old-cathedral'),
    ('hero', 'aurelia-hq'),
    ('generic', 'michigan-avenue-deco'),
    ('generic', 'glass-gable-house'),
]
avail = [(k, s) for (k, s) in showcase if os.path.isfile(os.path.join(MOD_OUT, k, f'{s}_lod0.gltf'))]
if avail:
    fig, axes = plt.subplots(len(avail), 2, figsize=(11, 3.6 * len(avail)), facecolor='#10131a')
    if len(avail) == 1:
        axes = [axes]
    for row, (k, s) in enumerate(avail):
        for col, (lbl, path) in enumerate([
            ('v3 Before', os.path.join(MOD_IN, k, f'{s}_lod0.gltf')),
            ('v4 After (Plajah Modeler + CUDA AO)', os.path.join(MOD_OUT, k, f'{s}_lod0.gltf')),
        ]):
            ax = axes[row][col]
            ax.set_facecolor('#10131a')
            ax.axis('off')
            a = plajah_arch_modeler.PlajahGltfAsset(path)
            pts, cols = [], []
            for p in a.prims:
                if len(p.pos) == 0:
                    continue
                pts.append(p.pos)
                if p.color is not None:
                    cols.append(p.color.astype(np.float32) / 255.0)
                else:
                    # Simple directional shade for v3 without AO
                    sh = np.clip(0.45 + 0.55 * (p.nrm @ np.array([0.4, 0.7, 0.58], dtype=np.float32)), 0.25, 1.0)
                    cols.append(np.stack([sh, sh, sh], axis=-1))
            V = np.concatenate(pts, axis=0)
            C = np.concatenate(cols, axis=0)
            # 3/4 architectural isometric view
            ang = 0.55
            px = V[:, 0] * np.cos(ang) + V[:, 2] * np.sin(ang)
            pz = -V[:, 0] * np.sin(ang) + V[:, 2] * np.cos(ang)
            py = V[:, 1] + pz * 0.22
            step = max(1, len(V) // 25000)
            order = np.argsort(pz[::step])
            ax.scatter(px[::step][order], py[::step][order], c=C[::step][order], s=1.0, linewidths=0)
            ax.set_aspect('equal')
            ax.set_title(f'{k}/{s} — {lbl} ({a.total_tris():,} tris)', color='white', fontsize=9.5)
    plt.tight_layout()
    geo_qa = os.path.join(WORK_OUT, 'geometry_qa_preview.png')
    plt.savefig(geo_qa, dpi=135, facecolor='#10131a')
    plt.show()
    print('Stage 2 Geometry QA saved:', geo_qa)

In [ ]:
# Cell 5: Package ironport_v4_outputs.zip (save to Google Drive or trigger browser download)
import json, os, shutil, zipfile

# Write clean output manifest for local ingest_ironport_colab.mjs
out_models = []
for kind in ('hero', 'generic'):
    d = os.path.join(WORK_OUT, 'models-v4', kind)
    if os.path.isdir(d):
        for fn in sorted(os.listdir(d)):
            if fn.endswith('_lod0.gltf'):
                out_models.append(f"{kind}/{fn.replace('_lod0.gltf', '')}")

out_textures = []
for sub in ('shared', 'hero-facades'):
    d = os.path.join(WORK_OUT, 'textures-v4', sub)
    if os.path.isdir(d):
        for fn in sorted(os.listdir(d)):
            if fn.endswith('_base.jpg'):
                out_textures.append(fn.replace('_base.jpg', ''))

out_manifest = {
    'version': 4,
    'modeler': 'Plajah Native Architectural Modeler v4 (Zero Blender)',
    'models': out_models,
    'textures': out_textures,
    'hasOrm': True,
    'hasHeight': True,
}
with open(os.path.join(WORK_OUT, 'v4-manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(out_manifest, f, indent=2)

ZIP_OUT = '/content/ironport_v4_outputs.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for root, _, fnames in os.walk(WORK_OUT):
        for fn in sorted(fnames):
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, WORK_OUT).replace('\\', '/')
            zf.write(full, rel)

mb = os.path.getsize(ZIP_OUT) / 1e6
print(f'READY: {ZIP_OUT} ({mb:.2f} MB) -> {len(out_models)} models, {len(out_textures)} PBR texture sets')

if USE_GOOGLE_DRIVE:
    drive_dst = os.path.join(DRIVE_DIR, 'ironport_v4_outputs.zip')
    shutil.copyfile(ZIP_OUT, drive_dst)
    print('Copied to Google Drive:', drive_dst)
else:
    from google.colab import files
    files.download(ZIP_OUT)